In [220]:
%pip install datasets scikit-learn


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [221]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [222]:
dataset = load_dataset('sms_spam')
dataset

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

In [223]:
dataset = dataset['train'].train_test_split(test_size=0.2) # Datensatz split
dataset["train"]

Dataset({
    features: ['sms', 'label'],
    num_rows: 4459
})

In [224]:
dataset["train"]["sms"][9]

'No calls..messages..missed calls\n'

In [225]:
c = -1
for i in dataset["train"]["label"]:
    c+=1
    if i == 1:
        break
c

2

In [226]:
dataset["train"]["label"][9]

0

In [227]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer( binary=False) 
X_train = vectorizer.fit_transform([x['sms'] for x in dataset['train']]).toarray()
X_test = vectorizer.transform([x['sms'] for x in dataset['test']]).toarray()
y_train = [x['label'] for x in dataset['train']]
y_test = [x['label'] for x in dataset['test']]

In [228]:
import torch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(dataset, batch_size=64, shuffle=False)




In [229]:
import torch.nn as nn
import torch.optim as optim


class SpamClassifier(nn.Module): 
    
    def __init__(self, input_dim): # Konstruktor 
        super().__init__() # Initialisieren der Superklasse
        # FC-FFNN - Fully connected feed-forward 
        self.fc1 = nn.Linear(input_dim, 32) # IMMER Zweierpotenzen 
        self.fc2 = nn.Linear(32, 1) # binäre Klassifikation

    def forward(self, x ): # = ich wende NN auf mein Input an, Forward-Pass
        x = self.fc1(x) # Matrizenmultiplikation
        x = torch.relu(x) # Aktivierung mit ReLU
        x = self.fc2(x) 
        x = torch.sigmoid(x)
        return x 

class SpamClassifier(nn.Module): 
    
    def __init__(self, input_dim): # Konstruktor 
        super().__init__() 
        self.model = nn.Sequential(
            nn.Linear(input_dim, 32),
            #nn.BatchNorm1d(32), # Preactivation
            nn.ReLU(),
            nn.BatchNorm1d(32), # Postactivation 
            nn.Linear(32, 1),
            nn.Dropout(0.3),
            nn.Sigmoid()
        )

    def forward(self, x): return self.model(x)  

In [230]:
model = SpamClassifier(input_dim=X_train.shape[1])
n_epochs = 10
#optimizer = optim.SGD(model.parameters(), lr = 1e-2)
optimizer = optim.Adam(model.parameters(), lr = 1e-3)
lossfn = nn.BCELoss() # weil binär 

for epoch in range(n_epochs): # 10 Epochen Trainingloop 
    running_loss = 0.0 
    for xb,yb in train_loader:
        model.train() # Modell ins Trainingsmodus versetzen 
        optimizer.zero_grad() # Gradient am Anfang der Epoche nullen 
        output = model(xb) # Batch 
        loss = lossfn(output, yb) # Kosten
        loss.backward()
        optimizer.step() 
        running_loss += loss
    print(f"Epoch [{epoch+1}/{n_epochs}], Loss: {running_loss}")


Epoch [1/10], Loss: 31.20607566833496
Epoch [2/10], Loss: 21.25234031677246
Epoch [3/10], Loss: 17.412931442260742
Epoch [4/10], Loss: 16.519542694091797
Epoch [5/10], Loss: 15.26495361328125
Epoch [6/10], Loss: 14.489228248596191
Epoch [7/10], Loss: 14.820469856262207
Epoch [8/10], Loss: 14.493374824523926
Epoch [9/10], Loss: 14.67690372467041
Epoch [10/10], Loss: 15.28129768371582


In [231]:
model.eval() # Evaluierungsmodus
with torch.no_grad(): # Gradient wird nicht mehr benötigt 
    preds = model(X_test_tensor) # Wahrscheinlichkeiten 
    predicted = (preds > 0.5).int()
    acc = (predicted.squeeze() == y_test_tensor.squeeze().int()).float().mean().item() 
    print(f'Test set accuracy: {acc*100:.2f}%')

Test set accuracy: 99.46%


In [232]:
def predict_spam(msg):
    x = vectorizer.transform([msg]).toarray()
    x_tensor = torch.tensor(x, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        pred = model(x_tensor).item()
        return 'SPAM' if pred > 0.5 else 'Kein Spam'

print(predict_spam('Congratulations, you have won a prize!'))
print(predict_spam('Are we meeting at 7pm for dinner?'))

Kein Spam
Kein Spam


In [233]:
(sum(X_train_tensor[0]) / 1000)*100

tensor(0.5000)

In [234]:
len(X_train_tensor[0])

7800